In [2]:
# import des bibliothèques nécessaires
import numpy as np
import matplotlib.pyplot as plt
import pandas as pd
import seaborn as sns
from sklearn.metrics import confusion_matrix, ConfusionMatrixDisplay, classification_report
from sklearn.model_selection import train_test_split
from sklearn.preprocessing import StandardScaler
from sklearn.decomposition import PCA
from sklearn.cross_decomposition import PLSRegression
from sklearn.metrics import r2_score, root_mean_squared_error
from sklearn.linear_model import Ridge
from sklearn.ensemble import RandomForestRegressor


In [ ]:
# Récupération du dataset des features et cibles
df_features = pd.read_csv('data/df_features.csv')
df_features.head()

,BuildingType,PrimaryPropertyType,Neighborhood,YearBuilt,NumberofBuildings,NumberofFloors,PropertyGFABuilding(s),LargestPropertyUseType,LargestPropertyUseTypeGFA,SecondLargestPropertyUseType,SecondLargestPropertyUseTypeGFA,ThirdLargestPropertyUseType,ThirdLargestPropertyUseTypeGFA,ENERGYSTARScore,SiteEnergyUse(kBtu),TotalGHGEmissions
0,NonResidential,Hotel,DOWNTOWN,1927,1.0,12.0,88434,Hotel,88434.0,Aucun,0.0,Aucun,0.0,60.0,7226362.5,249.98
1,NonResidential,Hotel,DOWNTOWN,1996,1.0,11.0,88502,Hotel,83880.0,Parking,15064.0,Restaurant,4622.0,61.0,8387933.0,295.86
2,NonResidential,Hotel,DOWNTOWN,1969,1.0,41.0,759392,Hotel,756493.0,Aucun,0.0,Aucun,0.0,43.0,72587024.0,2089.28
3,NonResidential,Hotel,DOWNTOWN,1926,1.0,10.0,61320,Hotel,61320.0,Aucun,0.0,Aucun,0.0,56.0,6794584.0,286.43
4,NonResidential,Hotel,DOWNTOWN,1980,1.0,18.0,113580,Hotel,123445.0,Parking,68009.0,Swimming Pool,0.0,75.0,14172606.0,505.01


# Plusieurs datasets pour tester nos modèles


## Uniquement les valeurs numériques

In [ ]:
df_num = df_features.drop(columns=["BuildingType","PrimaryPropertyType","Neighborhood","LargestPropertyUseType","SecondLargestPropertyUseType","ThirdLargestPropertyUseType"])
df_num.info()

## Valeurs numériques + non numérique

In [ ]:
df_all = df_features
df_all.info()

## Valeurs numériques + non numériques réduites

In [ ]:
df_reduced = df_features.drop(columns = ["Neighborhood","YearBuilt","SecondLargestPropertyUseType","ThirdLargestPropertyUseType","ENERGYSTARScore"])
df_reduced.info()

## OneHotEncoder

On se retrouve avec 15 colonnes, il faut néansmoins encoder les colonnes qui ne sont pas des colonnes numériques : 
BuildingType - 
PrimaryPropertyType - 
Naigborhood -
LargestPropertyUseType -
SecondLargestPropertyUseType -
ThirdLargestPropertyUseType




In [9]:
# Choix du dataset
df_choisi = df_reduced

columns = df_choisi.select_dtypes(exclude="number").columns.tolist()
df_encoded = pd.get_dummies(df_choisi,columns = columns, drop_first=True, dtype=int)

df_encoded.head()

,NumberofBuildings,NumberofFloors,PropertyGFABuilding(s),LargestPropertyUseTypeGFA,SecondLargestPropertyUseTypeGFA,ThirdLargestPropertyUseTypeGFA,SiteEnergyUse(kBtu),TotalGHGEmissions,BuildingType_NonResidential,BuildingType_Nonresidential COS,...,LargestPropertyUseType_Restaurant,LargestPropertyUseType_Retail Store,LargestPropertyUseType_Self-Storage Facility,LargestPropertyUseType_Senior Care Community,LargestPropertyUseType_Social/Meeting Hall,LargestPropertyUseType_Strip Mall,LargestPropertyUseType_Supermarket/Grocery Store,LargestPropertyUseType_Urgent Care/Clinic/Other Outpatient,LargestPropertyUseType_Wholesale Club/Supercenter,LargestPropertyUseType_Worship Facility
0,1.0,12.0,88434,88434.0,0.0,0.0,7226362.5,249.98,1,0,...,0,0,0,0,0,0,0,0,0,0
1,1.0,11.0,88502,83880.0,15064.0,4622.0,8387933.0,295.86,1,0,...,0,0,0,0,0,0,0,0,0,0
2,1.0,41.0,759392,756493.0,0.0,0.0,72587024.0,2089.28,1,0,...,0,0,0,0,0,0,0,0,0,0
3,1.0,10.0,61320,61320.0,0.0,0.0,6794584.0,286.43,1,0,...,0,0,0,0,0,0,0,0,0,0
4,1.0,18.0,113580,123445.0,68009.0,0.0,14172606.0,505.01,1,0,...,0,0,0,0,0,0,0,0,0,0


On obtient 185 colonnes ! 
Il faut faire une réduction de dimension avant de pouvoir entrainer notre modèle. Mais tout d'abord, pour éviter d'avoir des poids différents dépendant de la taille des valeurs numériques, il faut Standardiser les valeurs

On veut uniquement scale nos feature alors on les définit avant de standardiser

In [124]:
# Features
X = df_encoded.drop(columns = ["SiteEnergyUse(kBtu)","TotalGHGEmissions"])
# Cible Energie
y_energy = df_encoded["SiteEnergyUse(kBtu)"]
# Cible emissions CO2
y_co2 = df_encoded["TotalGHGEmissions"]

X.info()

<class 'pandas.DataFrame'>
RangeIndex: 1542 entries, 0 to 1541
Columns: 183 entries, YearBuilt to ThirdLargestPropertyUseType_Worship Facility
dtypes: float64(6), int64(177)
memory usage: 2.2 MB


## Target Encoding

# Scaler

In [125]:
# Création du scaler afin de normaliser les données (résout le problème de poids des variables)
scaler = StandardScaler()

# Calcul des paramètres sur les données train
X_scaled = scaler.fit_transform(X)

X_scaled


array([[-1.05363584, -0.12011243,  1.22370322, ..., -0.08079242,
        -0.02547409, -0.02547409],
       [ 1.04822305, -0.12011243,  1.06587866, ..., -0.08079242,
        -0.02547409, -0.02547409],
       [ 0.22575653, -0.12011243,  5.80061526, ..., -0.08079242,
        -0.02547409, -0.02547409],
       ...,
       [ 0.37806515, -0.12011243, -0.51236687, ..., 12.37739876,
        -0.02547409, -0.02547409],
       [ 0.83499099, -0.12011243, -0.51236687, ..., -0.08079242,
        -0.02547409, -0.02547409],
       [-0.71855688, -0.12011243, -0.51236687, ..., -0.08079242,
        -0.02547409, -0.02547409]], shape=(1542, 183))

# Réduction de dimension

## Test ACP


In [126]:
# définition du reducteur de dimension (ACP)
pca = PCA()

# Application sur nos features
X_pca = pca.fit_transform(X_scaled)

X_pca

array([[ 5.97189779e-01,  7.48320901e-01, -1.62484527e+00, ...,
        -4.83706642e-16,  1.41168978e-16, -3.60865503e-16],
       [ 2.45360107e+00,  6.03164405e-01, -1.06594956e+00, ...,
         2.71465499e-15,  2.08827272e-16, -1.46363790e-16],
       [ 4.93159784e+00, -2.17377590e-01, -2.45430427e+00, ...,
        -5.80121781e-16, -2.04284154e-16, -1.14309158e-16],
       ...,
       [-1.77683899e+00, -1.06945079e+01,  2.01110811e+00, ...,
         2.84676469e-17,  1.13524108e-17, -3.24041673e-17],
       [-2.03075312e+00, -1.16486273e+01,  4.76958744e+00, ...,
         2.47680000e-17, -1.03190876e-17,  4.01899880e-17],
       [-2.14373791e+00, -1.15133652e+01,  4.85695459e+00, ...,
         1.13529670e-17,  6.20876888e-17,  2.59951302e-17]],
      shape=(1542, 183))

In [128]:
# Récuperer les axes qui expliquent 95% de variance cumulée 

variance_expliquee = pca.explained_variance_ratio_
variance_cumulee = np.cumsum(variance_expliquee)

for i, (var, cum) in enumerate(zip(variance_expliquee, variance_cumulee), 1):
    print(f"Axe PC{i} : {var:.2%} (Cumulé : {cum:.2%})")
    
# Méthode non adaptée, car au 26e axe --> que 33% de la variance


Axe PC1 : 2.80% (Cumulé : 2.80%)
Axe PC2 : 1.85% (Cumulé : 4.65%)
Axe PC3 : 1.64% (Cumulé : 6.30%)
Axe PC4 : 1.60% (Cumulé : 7.90%)
Axe PC5 : 1.36% (Cumulé : 9.26%)
Axe PC6 : 1.33% (Cumulé : 10.58%)
Axe PC7 : 1.30% (Cumulé : 11.89%)
Axe PC8 : 1.25% (Cumulé : 13.14%)
Axe PC9 : 1.23% (Cumulé : 14.37%)
Axe PC10 : 1.21% (Cumulé : 15.59%)
Axe PC11 : 1.21% (Cumulé : 16.80%)
Axe PC12 : 1.20% (Cumulé : 18.00%)
Axe PC13 : 1.19% (Cumulé : 19.19%)
Axe PC14 : 1.18% (Cumulé : 20.36%)
Axe PC15 : 1.16% (Cumulé : 21.53%)
Axe PC16 : 1.15% (Cumulé : 22.68%)
Axe PC17 : 1.14% (Cumulé : 23.82%)
Axe PC18 : 1.13% (Cumulé : 24.96%)
Axe PC19 : 1.13% (Cumulé : 26.09%)
Axe PC20 : 1.11% (Cumulé : 27.19%)
Axe PC21 : 1.09% (Cumulé : 28.28%)
Axe PC22 : 1.01% (Cumulé : 29.29%)
Axe PC23 : 0.97% (Cumulé : 30.26%)
Axe PC24 : 0.96% (Cumulé : 31.22%)
Axe PC25 : 0.88% (Cumulé : 32.10%)
Axe PC26 : 0.88% (Cumulé : 32.98%)
Axe PC27 : 0.85% (Cumulé : 33.83%)
Axe PC28 : 0.85% (Cumulé : 34.68%)
Axe PC29 : 0.83% (Cumulé : 35.51%)

Et si on ne garde que les colonnes de base numériques ? 

In [147]:
df_num = df.drop(columns=["BuildingType","PrimaryPropertyType","Neighborhood","LargestPropertyUseType","SecondLargestPropertyUseType","ThirdLargestPropertyUseType"])
df_num.info()

<class 'pandas.DataFrame'>
RangeIndex: 1542 entries, 0 to 1541
Data columns (total 10 columns):
 #   Column                           Non-Null Count  Dtype  
---  ------                           --------------  -----  
 0   YearBuilt                        1542 non-null   int64  
 1   NumberofBuildings                1542 non-null   float64
 2   NumberofFloors                   1542 non-null   float64
 3   PropertyGFABuilding(s)           1542 non-null   int64  
 4   LargestPropertyUseTypeGFA        1542 non-null   float64
 5   SecondLargestPropertyUseTypeGFA  1542 non-null   float64
 6   ThirdLargestPropertyUseTypeGFA   1542 non-null   float64
 7   ENERGYSTARScore                  1542 non-null   float64
 8   SiteEnergyUse(kBtu)              1542 non-null   float64
 9   TotalGHGEmissions                1542 non-null   float64
dtypes: float64(8), int64(2)
memory usage: 120.6 KB


In [149]:
# Features
X_num = df_num.drop(columns = ["SiteEnergyUse(kBtu)","TotalGHGEmissions"])
# Cible Energie
y_energy_num = df_num["SiteEnergyUse(kBtu)"]
# Cible emissions CO2
y_co2_num = df_num["TotalGHGEmissions"]

In [151]:
# Création du scaler afin de normaliser les données (résout le problème de poids des variables)
scaler = StandardScaler()

# Calcul des paramètres sur les données train
X_scaled_num = scaler.fit_transform(X_num)

X_scaled_num

array([[-1.05363584e+00, -1.20112425e-01,  1.22370322e+00, ...,
        -3.88266301e-01, -2.11165661e-01, -2.50982303e-01],
       [ 1.04822305e+00, -1.20112425e-01,  1.06587866e+00, ...,
        -8.69829344e-02,  1.05630742e-01, -2.08378306e-01],
       [ 2.25756531e-01, -1.20112425e-01,  5.80061526e+00, ...,
        -3.88266301e-01, -2.11165661e-01, -9.75250257e-01],
       ...,
       [ 3.78065146e-01, -1.20112425e-01, -5.12366870e-01, ...,
        -2.76785056e-01, -2.11165661e-01,  2.60265664e-01],
       [ 8.34990992e-01, -1.20112425e-01, -5.12366870e-01, ...,
        -2.58244848e-01, -1.77991827e-01,  4.64168064e-03],
       [-7.18556884e-01, -1.20112425e-01, -5.12366870e-01, ...,
        -2.28264513e-01, -1.35222256e-01,  4.64168064e-03]],
      shape=(1542, 8))

In [152]:
# définition du reducteur de dimension (ACP)
pca = PCA()

# Application sur nos features
X_pca_num = pca.fit_transform(X_scaled_num)

# Récuperer les axes qui expliquent 95% de variance cumulée 

variance_expliquee = pca.explained_variance_ratio_
variance_cumulee = np.cumsum(variance_expliquee)

for i, (var, cum) in enumerate(zip(variance_expliquee, variance_cumulee), 1):
    print(f"Axe PC{i} : {var:.2%} (Cumulé : {cum:.2%})")

Axe PC1 : 43.75% (Cumulé : 43.75%)
Axe PC2 : 13.34% (Cumulé : 57.10%)
Axe PC3 : 12.68% (Cumulé : 69.77%)
Axe PC4 : 11.32% (Cumulé : 81.10%)
Axe PC5 : 9.59% (Cumulé : 90.69%)
Axe PC6 : 5.12% (Cumulé : 95.81%)
Axe PC7 : 3.79% (Cumulé : 99.60%)
Axe PC8 : 0.40% (Cumulé : 100.00%)


• n_components=4 (81,10 % de variance) : pour modèle très compact et éliminer un maximum de bruit.

• n_components=5 (90,69 % de variance) : plus sûr par défaut, il dépasse le seuil classique des 90 % tout en supprimant les 3 derniers axes peu utiles.

In [ ]:
# On refait mais cette fois-ci en pipeline


# 2. Découper en Train (80%) et Test (20%) AVANT toute transformation
X_train, X_test, y_train, y_test = train_test_split(X_num, y_energy_num, test_size=0.2, random_state=42)

# 3. Créer le Pipeline : Standardisation -> ACP (5 axes) -> Modèle de régression
# Exemple avec une régression linéaire régularisée (Ridge)
pipeline = Pipeline([
    ("scaler", StandardScaler()),
    ("pca", PCA(n_components=5)),
    ("modele", Ridge()) # Remplacez par RandomForestRegressor(random_state=42) pour tester du non-linéaire
])

# 4. Entraîner tout le pipeline sur le Train
pipeline.fit(X_train, y_train)

# 5. Évaluer sur le Test
y_pred = pipeline.predict(X_test)
print(f"R² sur le test : {r2_score(y_test, y_pred):.3f}")
print(f"Erreur moyenne (RMSE) : {root_mean_squared_error(y_test, y_pred):.3f}")

## PLS (Partial Least Squares) - Hybride - Réduction de dimension + Régression linéaire

In [ ]:
# 1. Séparation Train / Test
X_train, X_test, y_train, y_test = train_test_split(X_scaled, y_energy, test_size=0.2, random_state=42)

# 2. Réduction à k composantes + prédiction directe
pls = PLSRegression(n_components=9)
pls.fit(X_train, y_train)

# Pour récupérer vos colonnes réduites (si vous voulez les passer à un autre modèle) :
X_train_pls = pls.transform(X_train)
X_test_pls = pls.transform(X_test)

# Pour évaluer directement la performance de prédiction :
y_pred = pls.predict(X_test)
print(f"R² : {r2_score(y_test, y_pred):.3f}")
print(f"RMSE : {root_mean_squared_error(y_test, y_pred):.3f}")

R² : 0.761
RMSE : 9881625.145


# Modèles de régression

## Régression linéaire

La régression linéaire est une méthode statistique qui utilise des données de valeur connue pour prédire la valeur de données inconnues. La relation entre une ou plusieurs variables dépendantes et indépendantes est modélisée en ajustant une équation linéaire aux données observées. Les méthodes de régression linéaire excellent dans la détection de modèles dans les données historiques, offrant aux équipes marketing et commerciales une compréhension détaillée de la façon dont le comportement des clients, l'utilisation des services, la tarification et les données démographiques ont un impact sur les taux d'attrition. La régression linéaire multiple peut aider les entreprises à prédire l'attrition des clients en identifiant et en quantifiant les principaux facteurs incitant un client à partir.

## Régression polynomiale

La régression polynomiale est une forme avancée de régression linéaire utilisée pour capturer des modèles complexes dans les données. Elle modélise la relation entre les variables dépendantes et indépendantes sous la forme d'un polynôme de degré n. En ajustant une équation non linéaire aux données, elle peut capturer des relations non linéaires, ce qui la rend utile lors de l'utilisation d'ensembles de données complexes. Ce type de modèle de régression est couramment utilisé dans les applications de services financiers. Avec la capacité de capturer des interactions non linéaires entre des variables telles que l'âge, l'historique de conduite et le type de véhicule, la régression polynomiale permet aux assureurs de mieux évaluer les facteurs de risque et de prédire les résultats, ce qui se traduit par des décisions de souscription plus éclairées.

## Régression Ridge

La régression Ridge est une méthode de régularisation statistique utilisée pour corriger le surapprentissage sur les données d'entraînement des modèles de machine learning. La régression Ridge est un bon choix pour analyser la multicolinéarité, la présence de fortes intercorrélations entre deux ou plusieurs variables indépendantes au sein d'un modèle de régression multiple. Cela empêche le surapprentissage en ajoutant une pénalité aux coefficients de régression. Dans le domaine de la santé, la régression Ridge est utilisée pour identifier la relation entre un grand nombre de facteurs génétiques, liés au mode de vie et environnementaux, et le risque de développer des maladies spécifiques. Ce type de régression peut jouer un rôle important dans la création de modèles plus puissants et plus fiables pour prédire le risque de maladie individuelle en fonction de nombreux facteurs complexes et interdépendants.

## Régression Lasso

La régression Lasso (Least Absolute Shrinkage and Selection Operator) est une forme de régression linéaire qui utilise la contraction, les coefficients étant réduits vers un point central, comme la moyenne. Un cas d'usage principal de la régression Lasso est l'automatisation de la sélection de features. La régression Lasso sélectionne automatiquement les features utiles, éliminant les features inutiles ou redondantes.

## Régression elastic net

La régression elastic net fusionne les pénalités des régressions Lasso et Ridge, ce qui donne un modèle de régression en machine learning capable de trouver un équilibre entre la sélection de variables et la gestion de la multicolinéarité dans les modèles prédictifs. Dans le contexte de l'analyse sportive, la capacité de la régression elastic net à gérer un large éventail de variables corrélées (telles que les statistiques des joueurs, les mesures physiques et les conditions de jeu) la rend utile pour analyser la performance des joueurs et prédire les résultats des matchs.